<a href="https://colab.research.google.com/github/codewithUswaFatima/code-switching-codesaviours-si26-Uswa/blob/main/SI26_Uswa_Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 — Train Language ID Model + Deploy
### SI26 Code Saviours — Code Switching NLP (Uswa Fatima)

This notebook fine-tunes `xlm-roberta-base` as a **token classification** model
that labels each word in a Roman Urdu–English sentence as `URD`, `ENG`, or `MIX`,
using the `dataset.csv`.




In [1]:
!pip install -q transformers==4.44.2 torch datasets evaluate seqeval accelerate huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv('/content/dataset.csv')


label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}


grouped = df.groupby('sentence', sort=False)
sentences = [
    {'words': g['word'].tolist(), 'labels': g['label'].tolist()}
    for _, g in grouped
]

print(f'Total sentences: {len(sentences)}')
print(f'Total tokens: {sum(len(s["words"]) for s in sentences)}')


has_mix = ['MIX' if 'MIX' in s['labels'] else 'OTHER' for s in sentences]

train_data, test_data = train_test_split(
    sentences, test_size=0.2, random_state=42, stratify=has_mix
)

print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')
print(f'MIX-containing sentences in train: {sum(1 for s in train_data if "MIX" in s["labels"])}')
print(f'MIX-containing sentences in test:  {sum(1 for s in test_data if "MIX" in s["labels"])}')


Total sentences: 191
Total tokens: 2025
Training sentences: 152
Testing sentences: 39
MIX-containing sentences in train: 18
MIX-containing sentences in test:  4


In [3]:
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset
import numpy as np

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['words'], truncation=True,
                           is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)               # special tokens ([CLS]/[SEP]/pad)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])  # first sub-token of a word gets the label
            else:
                label_ids.append(-100)                # other sub-tokens of the same word are ignored
            prev_word = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized


def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data]
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

In [4]:
import evaluate as hf_evaluate
seqeval = hf_evaluate.load('seqeval')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    flat_metrics = {
        'overall_precision': results['overall_precision'],
        'overall_recall': results['overall_recall'],
        'overall_f1': results['overall_f1'],
        'overall_accuracy': results['overall_accuracy'],
    }
    for label in ['URD', 'ENG', 'MIX']:
        if label in results:
            flat_metrics[f'{label}_f1'] = results[label]['f1']
            flat_metrics[f'{label}_precision'] = results[label]['precision']
            flat_metrics[f'{label}_recall'] = results[label]['recall']
    return flat_metrics


In [5]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=15,                 # small dataset -> more epochs needed to converge
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',               # use 'evaluation_strategy' if transformers<4.41 complains
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='overall_f1',
    greater_is_better=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()
print('Training complete!')


Starting training...


Epoch,Training Loss,Validation Loss,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.772700,0.100876,0.943820,0.954545,0.949153,0.978313
2,0.096200,0.056954,0.969697,0.969697,0.969697,0.985542
3,0.044400,0.044998,0.966418,0.981061,0.973684,0.985542
4,0.015200,0.053951,0.973881,0.988636,0.981203,0.990361
5,0.019600,0.058707,0.973881,0.988636,0.981203,0.990361
6,0.010600,0.054216,0.973881,0.988636,0.981203,0.990361
7,0.011500,0.096662,0.963100,0.988636,0.975701,0.980723
8,0.002600,0.058441,0.973881,0.988636,0.981203,0.990361
9,0.001200,0.062930,0.970037,0.981061,0.975518,0.987952
10,0.006400,0.057335,0.973881,0.988636,0.981203,0.990361


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not t

Training complete!


In [6]:
final_metrics = trainer.evaluate()
for k, v in final_metrics.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')


eval_loss: 0.0540
eval_overall_precision: 0.9739
eval_overall_recall: 0.9886
eval_overall_f1: 0.9812
eval_overall_accuracy: 0.9904
eval_runtime: 0.1593
eval_samples_per_second: 244.7800
eval_steps_per_second: 31.3820
epoch: 15.0000


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


In [8]:
import torch

def predict(sentence_words):
    device = next(model.parameters()).device  # match wherever the model currently lives (cuda or cpu)
    inputs = tokenizer(sentence_words, is_split_into_words=True,
                        return_tensors='pt', truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    preds = torch.argmax(logits, dim=2)[0].cpu().tolist()
    word_ids = tokenizer(sentence_words, is_split_into_words=True, truncation=True).word_ids()

    results, prev_word = [], None
    for pred, word_id in zip(preds, word_ids):
        if word_id is None or word_id == prev_word:
            continue
        results.append((sentence_words[word_id], id2label[pred]))
        prev_word = word_id
    return results

test_sentence = ['Yaar', 'mujhe', 'thora', 'time', 'chahiye', 'for', 'this', 'assignment']
for word, label in predict(test_sentence):
    print(f'{word:12s} -> {label}')


Yaar         -> URD
mujhe        -> URD
thora        -> URD
time         -> ENG
chahiye      -> URD
for          -> ENG
this         -> ENG
assignment   -> ENG


In [26]:
from huggingface_hub import whoami

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxx"


print(whoami(token=HF_TOKEN))

repo_name = 'code-switching-codesaviours-si26-uswa'

model.push_to_hub(repo_name, token=HF_TOKEN)
tokenizer.push_to_hub(repo_name, token=HF_TOKEN)

print(f'Model published at: https://huggingface.co/122Uswa/{repo_name}')

{'type': 'user', 'id': '6a4234559b5a2d9c5a48ad99', 'name': '122Uswa', 'fullname': 'Uswa Fatima', 'email': 'uswa0464@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/4da06a9e8180a2184bade5ff8b3cc0e9.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab-write', 'role': 'write', 'createdAt': '2026-08-13T15:51:34.634Z'}}}


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3gy77ax/model.safetensors:   4%|4         | 48.0MB / 1.11GB            

No files have been modified since last commit. Skipping to prevent empty commit.


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...s/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...mpf4thba_s/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model published at: https://huggingface.co/122Uswa/code-switching-codesaviours-si26-uswa
